In [ ]:
import os
import csv
import time
import json
import math
import itertools
from dataclasses import dataclass, asdict
import numpy as np
import gc

import torch
import torch.nn as nn
from torch.nn import functional as F

# --- CENTRAL CONFIGURATION ---
@dataclass
class ExperimentConfig:
    run_name: str = "baseline"
    seed: int = 1337
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Standard Hyperparameters
    batch_size: int = 64
    block_size: int = 256
    max_iters: int = 50
    eval_interval: int = 250
    eval_iters: int = 200
    learning_rate: float = 1e-3
    vocab_size: int = 65

    #logging
    log_interval: int = 10 

    # Model Architecture
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2
    
    # --- ABLATION FLAGS ---
    norm_type: str = "layernorm"   # Options: 'layernorm', 'none'
    linear_type: str = "standard"  # Options: 'standard'
    use_pos_encoding: bool = True  # Options: True, False

#similar to original code
'''what was removed

Learning Rate Decay (Cosine Schedule): The original code doesn't just use a flat 1e-3 learning rate. It starts at 0 (warmup), goes up to 1e-3, and then slowly curves down to 1e-4 like a cosine wave.
Weight Decay Splitting: It applies weight_decay=0.1 to matrices, but weight_decay=0.0 to biases and LayerNorms. This version just applies standard AdamW to everything equally.
Gradient Clipping: The original code prevents exploding math by clipping gradients (grad_clip = 1.0). This code lets the gradients flow naturally.'''

"what was removed\n\nLearning Rate Decay (Cosine Schedule): The original code doesn't just use a flat 1e-3 learning rate. It starts at 0 (warmup), goes up to 1e-3, and then slowly curves down to 1e-4 like a cosine wave.\nWeight Decay Splitting: It applies weight_decay=0.1 to matrices, but weight_decay=0.0 to biases and LayerNorms. This version just applies standard AdamW to everything equally.\nGradient Clipping: The original code prevents exploding math by clipping gradients (grad_clip = 1.0). This code lets the gradients flow naturally."

In [61]:
#THIS IS WHERE NEW ARCHITECTURES GO ***HEREEEEE***
# 1. Modular Normalization Builder
def get_norm_layer(config, ndim):
    if config.norm_type == "layernorm":
        return nn.LayerNorm(ndim, bias=False)
    elif config.norm_type == "none":
        return nn.Identity() #does absolutely nothing
    else:
        raise ValueError(f"Unknown norm_type: {config.norm_type}")

# 2. Modular Linear Builder
def get_linear_layer(config, in_features, out_features, bias=False):
    if config.linear_type == "standard":
        return nn.Linear(in_features, out_features, bias=bias)
    # elif config.linear_type == "bitlinear": # QUANTISATION WILL BE ADDED LATER
    #     return BitLinear(in_features, out_features)
    else:
        raise ValueError(f"Unknown linear_type: {config.linear_type}")

In [62]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        
        # Uses our modular linear builder!
        self.c_attn = get_linear_layer(config, config.n_embd, 3 * config.n_embd)
        self.c_proj = get_linear_layer(config, config.n_embd, config.n_embd)
        
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v 
        
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = get_linear_layer(config, config.n_embd, 4 * config.n_embd)
        self.gelu   = nn.GELU()
        self.c_proj = get_linear_layer(config, 4 * config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Uses our modular norm builder!
        self.ln_1 = get_norm_layer(config, config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = get_norm_layer(config, config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd) if config.use_pos_encoding else None
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = get_norm_layer(config, config.n_embd)
        self.lm_head = get_linear_layer(config, config.n_embd, config.vocab_size)
        self.wte.weight = self.lm_head.weight

        # --- THE FIX: GPT-2 WEIGHT INITIALIZATION ---
        self.apply(self._init_weights)
        # Apply special scaled init to the residual projections
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if getattr(module, 'bias', None) is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        tok_emb = self.wte(idx)
        
        # Ablation: Add positional embeddings only if they exist
        if self.wpe is not None:
            pos = torch.arange(0, t, dtype=torch.long, device=device)
            pos_emb = self.wpe(pos)
            x = self.drop(tok_emb + pos_emb)
        else:
            x = self.drop(tok_emb)
            
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :]) 
            loss = None
        return logits, loss

In [63]:
def log_metrics_to_csv(config, metrics, filepath="metrics.csv"):
    """Saves all config settings and resulting metrics to a CSV."""
    # Combine config dict and metrics dict
    row_data = {**asdict(config), **metrics}
    
    file_exists = os.path.isfile(filepath)
    with open(filepath, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row_data.keys())
        if not file_exists:
            writer.writeheader() # Write column names on first run
        writer.writerow(row_data)

def train_run(config: ExperimentConfig):
    print(f"\n{'='*50}\nStarting Run: {config.run_name}\n{'='*50}")
    
    # 1. Perfect Reproducibility
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(config.seed)

    # 2. Data Loading
    train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
    val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')

    def get_batch(split):
        data = train_data if split == 'train' else val_data
        ix = torch.randint(len(data) - config.block_size, (config.batch_size,))
        x = torch.stack([torch.from_numpy((data[i:i+config.block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+1:i+1+config.block_size]).astype(np.int64)) for i in ix])
        return x.to(config.device), y.to(config.device)

    # 3. Model Setup
    model = GPT(config).to(config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    scaler = torch.amp.GradScaler('cuda', enabled=(config.device == 'cuda'))

    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(config.eval_iters)
            for k in range(config.eval_iters):
                X, Y = get_batch(split)
                _, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    # 4. Training Loop
    best_val_loss = float('inf')
    start_time = time.time()
    
    # We will save loss history as a json to plot graphs later
    loss_history = {'train': [], 'val': [], 'iter': []}

    for iter_num in range(config.max_iters + 1):
        if iter_num % config.eval_interval == 0:
            losses = estimate_loss()
            print(f"Step {iter_num:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")
            
            loss_history['iter'].append(iter_num)
            loss_history['train'].append(losses['train'])
            loss_history['val'].append(losses['val'])

        X, Y = get_batch('train')
        with torch.autocast(device_type=config.device, dtype=torch.float16 if config.device == 'cuda' else torch.bfloat16):
            logits, loss = model(X, Y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        
        #below does the logging
        if iter_num % config.log_interval == 0:
            print(f"iter {iter_num}: loss {loss.item():.4f}")
            
    end_time = time.time()
    train_time = round(end_time - start_time, 2)
    
    # 5. Save the weights
    torch.save(model.state_dict(), f"models/{config.run_name}.pt")
    
    # 6. Save the loss curves for later graphing
    with open(f"output/{config.run_name}_curves.json", 'w') as f:
        json.dump(loss_history, f)
        
    metrics = {
        "final_train_loss": loss_history['train'][-1],
        "final_val_loss": loss_history['val'][-1],
        "training_time_sec": train_time
    }
    
    # Log to CSV
    log_metrics_to_csv(config, metrics)
    print(f"Finished {config.run_name} in {train_time}s. Saved to models/ and output/metrics.csv")
    
    return model, metrics

In [64]:
# Define the variables you want to test
norm_options = ["layernorm", "none"]
pos_enc_options = [True, False]
head_options = [6, 1]

# Create an empty list to store our configurations
experiments = []

# 1. The Baseline
experiments.append(
    ExperimentConfig(run_name="baseline", norm_type="layernorm", use_pos_encoding=True, n_head=6)
)

# 2. Iterate through ablations
for norm, pos, heads in itertools.product(norm_options, pos_enc_options, head_options):
    if norm == "layernorm" and pos == True and heads == 6:
        continue 
    run_name = f"ablation_norm-{norm}_pos-{pos}_heads-{heads}"
    experiments.append(
        ExperimentConfig(
            run_name=run_name,
            norm_type=norm,
            use_pos_encoding=pos,
            n_head=heads
        )
    )
"""
# RUN ALL EXPERIMENTS
print(f"Queueing {len(experiments)} experiments")

for cfg in experiments:
    train_run(cfg)
    # Run Python garbage collector
    gc.collect()
    # Clear PyTorch CUDA cache
    torch.cuda.empty_cache()

"""

# Small single experiment
cfg = ExperimentConfig(
    run_name="Original Run",
    norm_type="layernorm",
    use_pos_encoding=True,
    n_head=6
)

train_run(cfg)
print("ALL EXPERIMENTS COMPLETED SUCCESSFULLY! Check output/metrics.csv")


Starting Run: Original Run
Step    0 | Train Loss: 4.2873 | Val Loss: 4.2822
iter 0: loss 4.2649
iter 10: loss 3.3996
iter 20: loss 3.3301
iter 30: loss 3.2983
iter 40: loss 3.3046
iter 50: loss 3.2912
Finished Original Run in 64.76s. Saved to models/ and output/metrics.csv
ALL EXPERIMENTS COMPLETED SUCCESSFULLY! Check output/metrics.csv


In [65]:
#IF THE CODE IS RUNNING SLOW
#ITS BECAUSE STUFF IS STILL IN VRAM
#run below to clear
#check vram in task manager
# Delete unused variables/models/tensors first
'''
del model
del optimizer
del loss
'''
# Run Python garbage collector
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()
